# 02 · SFT data → QLoRA → đánh giá base và LoRA

**Settings:** GPU **T4 x2** · Internet **On**

**Secrets** (Add-ons → Secrets), bật cho notebook này:
- `TEACHER_API_KEY`: API key của teacher LLM (dùng để tạo dữ liệu và làm judge)
- `HF_TOKEN` (không bắt buộc): tải model từ HuggingFace nhanh hơn

Thời gian ước tính:
- Tạo dữ liệu SFT: 20–60 phút (tuỳ giới hạn tốc độ của API)
- Train: 1–2 giờ
- Đánh giá: khoảng 40 phút

In [ ]:
REPO_URL = "https://github.com/sinhtruc24/Vietnamese-Legal-RAG.git"
WORK = "/kaggle/working/Vietnamese-Legal-RAG"

import os, socket
try:
    socket.create_connection(("github.com", 443), timeout=5).close()
except OSError:
    raise SystemExit("Không có Internet: Settings (bên phải) -> Internet = On, "
                     "rồi Run -> Restart & clear cell outputs và chạy lại.")

if not os.path.exists(WORK):
    !git clone -q {REPO_URL} {WORK}
else:
    !git -C {WORK} pull -q
assert os.path.exists(WORK), "Clone thất bại: kiểm tra REPO_URL và repo có để Public không"
%cd {WORK}
!git log --oneline -1

In [ ]:
!pip install -q -e ".[ml]" openai huggingface_hub
!pip install -q -r requirements-train.txt

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["TEACHER_API_KEY"] = secrets.get_secret("TEACHER_API_KEY")
os.environ["JUDGE_API_KEY"] = os.environ["TEACHER_API_KEY"]
try:
    os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
except Exception:
    pass

# ---- Teacher / judge: key và endpoint PHẢI cùng nhà cung cấp ----
# Gemini (key từ aistudio.google.com):
TEACHER_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
TEACHER_MODEL = "gemini-3.8-flash"      # hết quota free (lỗi 429) thì thử "gemini-3.1-flash-lite"
WORKERS = 2                             # free tier giới hạn số request/phút
# OpenAI thì đổi thành: TEACHER_BASE_URL = "https://api.openai.com/v1"; TEACHER_MODEL = "gpt-4o-mini"; WORKERS = 8

STUDENT_MODEL = "Qwen/Qwen2.5-3B-Instruct"

## 1. Dữ liệu và BM25 index
Chỉ cần BM25 để lấy hard negative, build mất khoảng 1 phút.

In [ ]:
!python scripts/download_data.py
!python scripts/build_index.py --bigrams --skip-dense --index-dir indexes/full

## 2. Tạo dữ liệu SFT
Chạy thử 30 câu trước và **đọc kỹ vài mẫu**, rồi mới chạy toàn bộ.
Script chạy tiếp được nếu bị ngắt giữa chừng.

In [ ]:
!python scripts/build_sft_data.py --index-dir indexes/full --limit 30 --workers {WORKERS} \
    --teacher-base-url {TEACHER_BASE_URL} --teacher-model {TEACHER_MODEL}

In [ ]:
import json
rows = [json.loads(l) for l in open("data/sft/raw.jsonl", encoding="utf-8")]
for r in rows[:3]:
    print("TYPE:", r["type"], "| gold:", r["gold"])
    print(r["prompt"][1]["content"][-300:])
    print("ANSWER:", r["completion"][0]["content"])
    print("=" * 100)

In [ ]:
# Toàn bộ khoảng 2.4K câu train
!python scripts/build_sft_data.py --index-dir indexes/full --workers {WORKERS} \
    --teacher-base-url {TEACHER_BASE_URL} --teacher-model {TEACHER_MODEL}

## 3. Train QLoRA

In [ ]:
!python training/train_qlora.py --model {STUDENT_MODEL} \
    --train data/sft/train.jsonl --val data/sft/val.jsonl \
    --output-dir outputs/qwen-legal-lora --epochs 2

In [ ]:
import json, pandas as pd
pd.DataFrame(json.load(open("outputs/qwen-legal-lora/eval_history.json")))[["step", "eval_loss"]]

## 4. Đánh giá: base, LoRA và teacher
Mọi mô hình nhận **cùng ngữ cảnh** (chế độ oracle, seed cố định), nên chênh lệch điểm chỉ phản ánh chất lượng của mô hình.

In [ ]:
JUDGE = f"--judge-base-url {TEACHER_BASE_URL} --judge-model {TEACHER_MODEL}"
LIMIT = 300

!python scripts/eval_generation.py --backend hf --llm-model {STUDENT_MODEL} --name base --limit {LIMIT} {JUDGE}
!python scripts/eval_generation.py --backend hf --llm-model {STUDENT_MODEL} \
    --adapter outputs/qwen-legal-lora/adapter --name lora --limit {LIMIT} {JUDGE}

In [ ]:
# Teacher làm mốc tham chiếu (gọi qua API)
os.environ["LLM_API_KEY"] = os.environ["TEACHER_API_KEY"]
!python scripts/eval_generation.py --llm-base-url {TEACHER_BASE_URL} --llm-model {TEACHER_MODEL} \
    --name teacher --limit {LIMIT} {JUDGE}

In [ ]:
import glob
summaries = [json.load(open(p)) for p in sorted(glob.glob("results/gen_*_summary.json"))]
cols = ["name", "answer_rate", "gold_cite_rate", "cite_precision", "invalid_cite_rate",
        "uncited_rate", "refusal_acc", "faithfulness", "latency_ms"]
pd.DataFrame(summaries)[cols].set_index("name")

## 5. Phân tích lỗi
Xem những câu LoRA trả lời sai để viết phần phân tích trong README. Nhà tuyển dụng thường đánh giá cao phần này.

In [ ]:
lora = [json.loads(l) for l in open("results/gen_lora.jsonl", encoding="utf-8")]
bad = [r for r in lora if (r["answerable"] and not r["cites_gold"]) or (not r["answerable"] and not r["refused"])]
print(f"{len(bad)} / {len(lora)} lỗi")
for r in bad[:5]:
    print("gold:", r["gold"], "| cited:", r["cited"], "| refused:", r["refused"])
    print(r["answer"][:500])
    print("-" * 100)

## 6. Lưu
**Save Version → Save & Run All**. Sau đó ở tab Output, tải về:
- `outputs/qwen-legal-lora/adapter/`: LoRA adapter, dùng cho vLLM/Docker
- `results/`: kết quả để điền README